In [ ]:
# SECTION 1: Setup, Imports, and Utilities
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
from IPython.display import display, Image as IPImage
import pandas as pd

# Add root to path so imports work in Colab
if '/content/drive/MyDrive/satellite-segmentation' not in sys.path:
    sys.path.insert(0, '/content/drive/MyDrive/satellite-segmentation')

from utils.config import load_config
from datasets import get_dataset
from models.uncertainty_factory import get_uncertainty_model
from legacy.factory import get_model as get_legacy_model
from metrics.calibration import CalibrationMetrics
from metrics.segmentation import SegmentationMetrics

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

def evaluate_model(model, dataloader, config, is_uncertain=False, device='cuda'):
    seg_metrics = SegmentationMetrics(
        num_classes=config.data.num_classes,
        ignore_index=config.data.ignore_index if hasattr(config.data, 'ignore_index') else None,
        class_names=dataloader.dataset.get_class_names()
    )
    cal_metrics = CalibrationMetrics(
        num_bins=15,
        num_classes=config.data.num_classes,
        ignore_index=config.data.ignore_index if hasattr(config.data, 'ignore_index') else None
    )
    with torch.no_grad():
        for images, masks in dataloader:
            images, masks = images.to(device), masks.to(device)
            if is_uncertain:
                output = model(images, return_uncertainty=True)
                probs, preds, uncertainty = output['prob'], output['pred'], output['uncertainty']
            else:
                logits = model(images)
                probs = torch.softmax(logits, dim=1)
                preds = torch.argmax(probs, dim=1)
                uncertainty = None
            seg_metrics.update(preds, masks)
            cal_metrics.update(probs, preds, masks, uncertainty)
    return seg_metrics, cal_metrics, {**seg_metrics.compute(return_per_class=True), **cal_metrics.compute()}

print('✓ Setup complete')

In [ ]:
# SECTION 2: Load Models and Data
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
config = load_config('configs/uncertain_segformer.yaml')

val_dataset = get_dataset(config, split='val')
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=2)
print(f'✓ Loaded {len(val_dataset)} validation samples')

baseline_model = get_legacy_model(config).to(device)
base_ckpt = Path('/content/drive/MyDrive/satellite-segmentation/checkpoints/baseline/best.pth')
if base_ckpt.exists():
    baseline_model.load_state_dict(torch.load(base_ckpt, map_location=device)['model_state_dict'])
    print('✓ Baseline model loaded')
baseline_model.eval()

unc_ckpt = Path('/content/drive/MyDrive/satellite-segmentation/checkpoints/segformer/best_miou.pth')
if unc_ckpt.exists():
    checkpoint = torch.load(unc_ckpt, map_location=device, weights_only=False)
    arch = checkpoint['config']['model'].get('arch', 'SegFormer') if 'config' in checkpoint else config.model.get('arch', 'SegFormer')
    enc = checkpoint['config']['model'].get('encoder', config.model.encoder) if 'config' in checkpoint else config.model.encoder
    uncertain_model = get_uncertainty_model(arch, enc, config.data.num_classes, False).to(device)
    uncertain_model.load_state_dict(checkpoint['model_state_dict'] if 'model_state_dict' in checkpoint else checkpoint)
    uncertain_model.eval()
    print('✓ Uncertain model loaded dynamically')
else:
    print(f'⚠️ Uncertain checkpoint not found at {unc_ckpt}')

In [ ]:
# SECTION 3: Model Performance Comparison
print('[1/2] Evaluating baseline model...')
baseline_seg, baseline_cal, baseline_results = evaluate_model(baseline_model, val_loader, config, False, device)

print('[2/2] Evaluating uncertain model...')
uncertain_seg, uncertain_cal, uncertain_results = evaluate_model(uncertain_model, val_loader, config, True, device)

df_comparison = pd.DataFrame({
    'Metric': ['mIoU', 'Accuracy', 'ECE', 'MCE', 'Brier Score', 'Mean F1'],
    'Baseline': [baseline_results['miou'], baseline_results['accuracy'], baseline_results['ece'], baseline_results['mce'], baseline_results['brier'], baseline_results['f1']],
    'Uncertain (Ours)': [uncertain_results['miou'], uncertain_results['accuracy'], uncertain_results['ece'], uncertain_results['mce'], uncertain_results['brier'], uncertain_results['f1']]
})

df_comparison['Improvement (%)'] = [
    (uncertain_results['miou'] - baseline_results['miou']) / baseline_results['miou'] * 100,
    (uncertain_results['accuracy'] - baseline_results['accuracy']) / baseline_results['accuracy'] * 100,
    (baseline_results['ece'] - uncertain_results['ece']) / baseline_results['ece'] * 100,
    (baseline_results['mce'] - uncertain_results['mce']) / baseline_results['mce'] * 100,
    (baseline_results['brier'] - uncertain_results['brier']) / baseline_results['brier'] * 100,
    (uncertain_results['f1'] - baseline_results['f1']) / baseline_results['f1'] * 100
]

display(df_comparison.style.format({'Baseline': '{:.4f}', 'Uncertain (Ours)': '{:.4f}', 'Improvement (%)': '{:+.2f}%'}).background_gradient(subset=['Improvement (%)'], cmap='RdYlGn', vmin=-10, vmax=50))

In [ ]:
# SECTION 4: Plotting (Calibration, Error, Confusion)
Path('outputs').mkdir(exist_ok=True)

baseline_cal.plot_reliability_diagram(save_path='outputs/baseline_reliability.png')
plt.close()
uncertain_cal.plot_reliability_diagram(save_path='outputs/uncertain_reliability.png')
plt.close()

uncertain_cal.plot_uncertainty_vs_error(save_path='outputs/uncertainty_vs_error.png')
plt.close()

baseline_seg.plot_confusion_matrix(save_path='outputs/baseline_confusion.png', normalize=True)
plt.close()
uncertain_seg.plot_confusion_matrix(save_path='outputs/uncertain_confusion.png', normalize=True)
plt.close()

print('✓ All analytical plots generated and saved to outputs/')
display(IPImage('outputs/uncertain_reliability.png', width=800))

In [ ]:
# SECTION 5: Summary and Export
ratio = 0.0
if 'uncertainty_correct' in uncertain_results and uncertain_results['uncertainty_correct'] > 0:
    ratio = uncertain_results['uncertainty_incorrect'] / uncertain_results['uncertainty_correct']

print('\nKEY FINDINGS SUMMARY:')
print(f"• mIoU improvement: {(uncertain_results['miou'] - baseline_results['miou']) / baseline_results['miou'] * 100:+.2f}%")
print(f"• ECE reduction: {(baseline_results['ece'] - uncertain_results['ece']) / baseline_results['ece'] * 100:.1f}%")
print(f"• Uncertainty on errors is {ratio:.1f}x higher than correct predictions")

all_results = {
    'baseline': {k: float(v) if isinstance(v, (np.floating, float)) else v for k, v in baseline_results.items() if not isinstance(v, list)},
    'uncertain': {k: float(v) if isinstance(v, (np.floating, float)) else v for k, v in uncertain_results.items() if not isinstance(v, list)}
}

with open('outputs/final_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

with open('outputs/comparison_table.tex', 'w') as f:
    f.write(df_comparison.to_latex(index=False, float_format='%.4f'))

print('\n✅ JSON and LaTeX exports complete!')